In [5]:
from pydantic import BaseModel, ValidationError, Field
from typing import Optional
import json

In [6]:
# 1. Definimos nuestro "Contrato" de datos con Pydantic
class RespuestaTecnica(BaseModel):
    encontrado: bool = Field(description="Indica si la info está en el manual")
    id_pieza: Optional[str] = Field(None, description="El ID de la pieza si se encuentra")
    instrucciones: str = Field(description="Pasos a seguir o mensaje de error")
    nivel_riesgo: int = Field(ge=0, le=5, description="Escala de riesgo del 0 al 5")

In [7]:
# 2. Contexto real que "recuperamos" de la DB
contexto_manual = """
La Batería X-100 debe cargarse a 220V. 
NUNCA abrir la carcasa sin protección térmica. 
Riesgo de explosión nivel 4.
"""

In [8]:
# 3. SIMULACIÓN DE RESPUESTA DEL LLM (Escenario A: Respuesta correcta)
llm_output_a = '''{
    "encontrado": true, 
    "id_pieza": "X-100", 
    "instrucciones": "Cargar a 220V con protección térmica.", 
    "nivel_riesgo": 4}'''

In [9]:
# 4. SIMULACIÓN DE RESPUESTA DEL LLM (Escenario B: El LLM alucina o falla el formato)
llm_output_b = '''{
    "encontrado": true, 
    "instrucciones": "Usa un cargador normal de móvil.", 
    "nivel_riesgo": 10}''' # ¡Riesgo 10 no existe en nuestro modelo!

In [10]:
def validar_respuesta(json_str):
    try:
        data = json.loads(json_str)
        respuesta = RespuestaTecnica(**data)
        return f"✅ VÁLIDO: {respuesta}"
    except (ValidationError, ValueError) as e:
        return f"❌ ERROR DE VALIDACIÓN: {e}"

In [11]:
print("Probando Escenario A (Correcto):")
print(validar_respuesta(llm_output_a))

Probando Escenario A (Correcto):
✅ VÁLIDO: encontrado=True id_pieza='X-100' instrucciones='Cargar a 220V con protección térmica.' nivel_riesgo=4


In [12]:
print("\nProbando Escenario B (Fallo de esquema/lógica):")
print(validar_respuesta(llm_output_b))


Probando Escenario B (Fallo de esquema/lógica):
❌ ERROR DE VALIDACIÓN: 1 validation error for RespuestaTecnica
nivel_riesgo
  Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
